# 🏭 Factory AI Predictive Maintenance — Run Demo in Colab
### Built with IBM watsonx · DEMO-MFG-001

**No installs on your computer. No Netlify account. No sign-ups.**
Everything — backend AND frontend — runs inside this Colab VM.
You get one clickable public URL that opens the full demo in your browser.

| Step | What happens | Time |
|---|---|---|
| Cell 1 | Clone the repo | ~30 sec |
| Cell 2 | Install Python backend packages | ~1 min |
| Cell 3 | Install Node.js 20 + build the React frontend | ~2–3 min |
| Cell 4 | Start backend + frontend servers + open public tunnel | ~30 sec |
| Cell 5 | Verify everything works | ~15 sec |

### ▶️ How to run
Click **Runtime → Run all** (`Ctrl+F9`) — then wait ~4 minutes.
Cell 4 prints a link like `https://xyz.trycloudflare.com` — click it.

> ⚠️ Colab VMs reset after ~12 hours of inactivity. If the link stops working, just run **Runtime → Run all** again (takes ~2 min the second time thanks to caching).

---

## Cell 1 — Clone the repo

In [ ]:
import os, subprocess

# ── EDIT THIS if you forked the repo ──────────────────────────────────────────
REPO_URL = "https://github.com/Kgotla-AI-Demos/IBM-Pre-Sales-Demo-Builder.git"
# If the repo is private, use a Personal Access Token (PAT):
# REPO_URL = "https://YOUR_PAT@github.com/Kgotla-AI-Demos/IBM-Pre-Sales-Demo-Builder.git"
# Create a PAT at: https://github.com/settings/tokens  (tick the 'repo' scope)
# ──────────────────────────────────────────────────────────────────────────────

REPO_DIR  = "/content/repo"
DEMO_DIR  = f"{REPO_DIR}/demos/new-demo"

if not os.path.exists(REPO_DIR):
    print("→ Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("→ Repo already present — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

print(f"\n✅ Repo ready at {DEMO_DIR}")
os.listdir(DEMO_DIR)

## Cell 2 — Install Python backend dependencies

In [ ]:
%%bash
set -e
DEMO_DIR="/content/repo/demos/new-demo"

echo "→ Installing backend Python packages (uses Colab cache after first run)..."
pip install -q -r "$DEMO_DIR/backend/requirements.txt"
echo "✅ Backend packages ready"

## Cell 3 — Install Node.js 20 + build the React frontend

> This cell takes ~2–3 minutes the first time. Subsequent runs use the npm cache and take ~1 minute.

In [ ]:
%%bash
set -e
DEMO_DIR="/content/repo/demos/new-demo"
FRONTEND_DIR="$DEMO_DIR/frontend"

# ── Install Node.js 20 LTS (idempotent) ───────────────────────────────────────
if ! command -v node &>/dev/null || [[ $(node --version | cut -d. -f1 | tr -d 'v') -lt 20 ]]; then
    echo "→ Installing Node.js 20 LTS..."
    curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1
    apt-get install -y nodejs >/dev/null 2>&1
fi
echo "   Node $(node --version)  |  npm $(npm --version)"

# ── Write frontend .env with localhost API URL ─────────────────────────────────
# The frontend will talk to the backend on port 8000, both served through
# the same Cloudflare tunnel via a /api reverse-proxy path (see Cell 4).
cat > "$FRONTEND_DIR/.env" <<'EOF'
VITE_API_BASE_URL=/api
VITE_DEMO_TITLE=Factory AI Predictive Maintenance — Built with IBM watsonx
VITE_DEMO_CLIENT_CODE=DEMO-MFG-001
EOF

# ── npm install ───────────────────────────────────────────────────────────────
echo "→ Running npm install (uses cache if available)..."
cd "$FRONTEND_DIR"
npm install --prefer-offline --silent
echo "   npm install complete"

# ── Vite build ────────────────────────────────────────────────────────────────
echo "→ Building React app (vite build)..."
npm run build 2>&1

DIST_SIZE=$(du -sh "$FRONTEND_DIR/dist" | cut -f1)
echo ""
echo "✅ Frontend built successfully!  dist/ size: $DIST_SIZE"

## Cell 4 — Start backend + frontend + open public tunnel

This cell:
1. Starts the **FastAPI backend** on port 8000
2. Starts a **static file server** for the built React app on port 3000
3. Starts an **nginx-like Python proxy** on port 80 that routes `/api/*` → backend and everything else → frontend
4. Opens a **Cloudflare Tunnel** — prints your public URL

> Click the URL printed at the bottom — the full demo opens in your browser.

In [ ]:
import subprocess, threading, time, re, sys, os, signal

DEMO_DIR     = "/content/repo/demos/new-demo"
BACKEND_DIR  = f"{DEMO_DIR}/backend"
FRONTEND_DIR = f"{DEMO_DIR}/frontend"
BACKEND_PORT = 8000
STATIC_PORT  = 3000
PROXY_PORT   = 5000   # the single port exposed via Cloudflare

# ── Kill any previous processes on those ports ────────────────────────────────
for port in [BACKEND_PORT, STATIC_PORT, PROXY_PORT]:
    subprocess.run(f"fuser -k {port}/tcp 2>/dev/null || true", shell=True)
time.sleep(1)

# ── Write backend .env ────────────────────────────────────────────────────────
with open(f"{BACKEND_DIR}/.env", "w") as f:
    f.write("DEMO_MODE=mock\nCORS_ORIGINS=*\n")

# ── 1. Start FastAPI backend ──────────────────────────────────────────────────
print(f"→ Starting FastAPI backend on port {BACKEND_PORT}...")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", str(BACKEND_PORT)],
    cwd=BACKEND_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f"   Backend PID: {backend_proc.pid}")

# ── 2. Write a tiny Python reverse-proxy (no extra packages needed) ───────────
proxy_script = f"""
import http.server, urllib.request, urllib.error
import socketserver, os, shutil

BACKEND  = "http://127.0.0.1:{BACKEND_PORT}"
STATIC   = "{FRONTEND_DIR}/dist"
PORT     = {PROXY_PORT}

class ProxyHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a): pass          # silence access log

    def _forward_to_backend(self):
        url  = BACKEND + self.path
        body = None
        if self.command in ("POST", "PUT", "PATCH"):
            length = int(self.headers.get("Content-Length", 0))
            body   = self.rfile.read(length) if length else None
        hdrs = {{k: v for k, v in self.headers.items()
                 if k.lower() not in ("host", "connection", "transfer-encoding")}}
        try:
            req = urllib.request.Request(url, data=body, headers=hdrs, method=self.command)
            with urllib.request.urlopen(req, timeout=30) as resp:
                self.send_response(resp.status)
                for k, v in resp.headers.items():
                    if k.lower() not in ("transfer-encoding", "connection"):
                        self.send_header(k, v)
                self.send_header("Access-Control-Allow-Origin", "*")
                self.end_headers()
                shutil.copyfileobj(resp, self.wfile)
        except urllib.error.HTTPError as e:
            body = e.read()
            self.send_response(e.code)
            for k, v in e.headers.items():
                if k.lower() not in ("transfer-encoding", "connection"):
                    self.send_header(k, v)
            self.end_headers()
            self.wfile.write(body)
        except Exception as exc:
            self.send_response(502)
            self.end_headers()
            self.wfile.write(str(exc).encode())

    def _serve_static(self):
        path = self.path.split("?")[0].lstrip("/")
        filepath = os.path.join(STATIC, path)
        # SPA fallback: any path that isn't a real file → index.html
        if not os.path.isfile(filepath):
            filepath = os.path.join(STATIC, "index.html")
        ext = os.path.splitext(filepath)[1]
        mime = {{".html": "text/html", ".js": "application/javascript",
                ".css": "text/css", ".json": "application/json",
                ".svg": "image/svg+xml", ".png": "image/png",
                ".ico": "image/x-icon", ".woff2": "font/woff2"}}.get(ext, "application/octet-stream")
        try:
            with open(filepath, "rb") as fh:
                data = fh.read()
            self.send_response(200)
            self.send_header("Content-Type", mime)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except FileNotFoundError:
            self.send_response(404)
            self.end_headers()

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.end_headers()

    def do_GET(self):
        if self.path.startswith("/api"):
            self._forward_to_backend()
        else:
            self._serve_static()

    def do_POST(self):
        self._forward_to_backend()

socketserver.TCPServer.allow_reuse_address = True
with socketserver.TCPServer(("", PORT), ProxyHandler) as httpd:
    print(f"Proxy listening on port {{PORT}}", flush=True)
    httpd.serve_forever()
"""

proxy_path = "/tmp/demo_proxy.py"
with open(proxy_path, "w") as f:
    f.write(proxy_script)

print(f"→ Starting combined proxy server on port {PROXY_PORT}...")
proxy_proc = subprocess.Popen(
    [sys.executable, proxy_path],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(2)
print(f"   Proxy PID: {proxy_proc.pid}")

# ── 3. Install cloudflared (once) ─────────────────────────────────────────────
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("→ Downloading cloudflared (one-time ~20 MB)...")
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"
    ], check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    print("   cloudflared installed")

# ── 4. Start Cloudflare Tunnel ────────────────────────────────────────────────
print("→ Opening Cloudflare Tunnel (free, no account needed)...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://localhost:{PROXY_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

DEMO_URL = None
print("   Waiting for tunnel URL", end="", flush=True)
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        DEMO_URL = match.group(0)
        break
    print(".", end="", flush=True)
    time.sleep(1)

print()
if DEMO_URL:
    print()
    print("=" * 65)
    print("  ✅  YOUR DEMO IS LIVE")
    print()
    print(f"  🌐  {DEMO_URL}")
    print()
    print("  Click the link above — the demo opens in your browser.")
    print("  Share it with anyone. Works on any device.")
    print()
    print(f"  API docs:  {DEMO_URL}/docs    (backend Swagger)")
    print("=" * 65)

    import builtins
    builtins.DEMO_PUBLIC_URL = DEMO_URL
else:
    print("❌ Could not get tunnel URL — re-run this cell.")

## Cell 5 — Verify all endpoints are responding

> All checks should show ✅. If any show ❌, re-run Cell 4.

In [ ]:
import requests, builtins, json

BASE = getattr(builtins, 'DEMO_PUBLIC_URL', None)
if not BASE:
    BASE = input("Paste the URL printed by Cell 4 (e.g. https://xyz.trycloudflare.com): ").strip()

print("─" * 65)
print("  Running verification checks...")
print("─" * 65)

checks = [
    ("Frontend (index.html)",  BASE + "/",                  "GET",  None),
    ("Health",                 BASE + "/api/health",         "GET",  None),
    ("Dashboard data",         BASE + "/api/data/dashboard", "GET",  None),
    ("Equipment list",         BASE + "/api/data/equipment", "GET",  None),
    ("Work orders",            BASE + "/api/data/work-orders","GET", None),
    ("Analytics",              BASE + "/api/data/analytics", "GET",  None),
    ("AI models",              BASE + "/api/ai/models",      "GET",  None),
]

all_ok = True
for name, url, method, body in checks:
    try:
        fn   = requests.get if method == "GET" else requests.post
        resp = fn(url, json=body, timeout=20)
        icon = "✅" if resp.status_code == 200 else "❌"
        if resp.status_code != 200:
            all_ok = False
        print(f"  {icon}  {name:<28}  {resp.status_code}")
    except Exception as e:
        print(f"  ❌  {name:<28}  ERROR: {e}")
        all_ok = False

# AI analyze (POST)
print()
print("  Testing AI analyze endpoint (mock mode)...")
try:
    resp = requests.post(
        BASE + "/api/ai/analyze",
        json={"equipment_id": "EQ-0012", "scenario": "bearing_failure"},
        timeout=30,
    )
    if resp.status_code == 200 and "analysis" in resp.json():
        d = resp.json()
        print(f"  ✅  AI analyze                    200  confidence={d.get('confidence')}  mode={d.get('mode')}")
    else:
        print(f"  ❌  AI analyze                    {resp.status_code}  {resp.text[:200]}")
        all_ok = False
except Exception as e:
    print(f"  ❌  AI analyze                    ERROR: {e}")
    all_ok = False

print()
print("─" * 65)
if all_ok:
    print("  ✅  All checks passed!")
    print()
    print(f"  Open this URL in your browser:  {BASE}")
else:
    print("  ❌  Some checks failed — re-run Cell 4 to restart the servers.")
print("─" * 65)

---
## 🔁 After a Colab session restart (every ~12 hours)

The Colab VM resets when idle. When the link stops working:

1. Click **Runtime → Run all** (`Ctrl+F9`) — takes ~2 min (npm & pip caches)
2. Cell 4 prints a **new** tunnel URL — use that

---

## 💡 Tips

| Tip | Detail |
|---|---|
| Keep Colab tab open | The backend runs as long as Colab is running. Close it = demo goes offline |
| Prevent idle disconnect | Move your mouse in the Colab tab every few hours, or use Colab Pro |
| Share the demo | Give the Cloudflare URL to anyone — no login required |
| Live IBM watsonx.ai | In Cell 4's `.env` block set `DEMO_MODE=live` and add your `WATSONX_API_KEY` & `WATSONX_PROJECT_ID` |
| Demo presentation flow | Dashboard → Problem → AI Agent → Results → Architecture (15 min) |

---
*Built with IBM watsonx · DEMO-MFG-001 · IBM Pre-Sales Demo Builder*